In [4]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, numpy as np
from elo import add_elo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

df = pd.read_csv("../data/processed/laliga_matches.csv", parse_dates=["Date"], dtype={"Season": str})
df = add_elo(df)
df["y"] = df["FTR"].map({"H": 0, "D": 1, "A": 2})

train = df[df["Season"].isin(["1617","1718","1819","1920","2021","2122","2223"])]
test  = df[df["Season"].isin(["2324","2425","2526"])]

X_tr, y_tr = train[["elo_diff"]], train["y"]
X_te, y_te = test[["elo_diff"]], test["y"]

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_tr, y_tr)
P = modelo.predict_proba(X_te)

print("Log-loss Elo + logística:", round(log_loss(y_te, P, labels=[0,1,2]), 4))

# Referencias sobre el MISMO test, sin fuga
freq = np.bincount(y_tr) / len(y_tr)
P_base = np.tile(freq, (len(y_te), 1))
print("Log-loss línea base (freq. de train):", round(log_loss(y_te, P_base, labels=[0,1,2]), 4))

m = test.dropna(subset=["AvgCH","AvgCD","AvgCA"])
inv = 1 / m[["AvgCH","AvgCD","AvgCA"]].values
P_mkt = inv / inv.sum(axis=1, keepdims=True)
print("Log-loss mercado (mismo test):", round(log_loss(m["y"], P_mkt, labels=[0,1,2]), 4))

Log-loss Elo + logística: 0.9746
Log-loss línea base (freq. de train): 1.065
Log-loss mercado (mismo test): 0.9537


## Baseline: Elo + regresión logística

**Diseño (sin fuga de datos):**
- El rating Elo de cada equipo se calcula únicamente con partidos anteriores
  (se guarda el rating antes de actualizarlo con el resultado).
- Entrenamiento: temporadas 2016/17 a 2022/23. Test: 2023/24 a 2025/26
  (1.140 partidos).
- Parámetros de Elo fijados a mano (K = 20, ventaja local = 60,
  regresión a la media = 25 % por temporada), sin optimizar.

**Resultados (log-loss, menor es mejor):**

| Modelo | Log-loss |
|---|---|
| Línea base (frecuencias de train) | 1,0650 |
| Elo + logística | 0,9746 |
| Mercado (AvgC) | 0,9537 |

**Lectura:** con una sola variable, el Elo recupera ≈ 81 % de la distancia
entre la línea base y el mercado. La brecha restante es 0,021. Falta
cuantificar su incertidumbre y validar con varios cortes temporales.